# Assay Ground Truth: TADP Stratification with Nested Training Splits

This notebook creates stratified data splits for GEPA prompt optimisation experiments.

## Design Overview

**Stratification Metric:** Total Assay Data Points (TADP)
- Counts extractable assay values from `isolates_with_linking` and `no_isolates_only_assayinformation`
- Does NOT count `isolate_without_linking` (contains only IDs, no assay data)

**Split Structure:** Nested Training Splits (10% ⊂ 20% ⊂ 30%)
- Enables fair comparison of training data requirements
- Eliminates confounding variance from sample composition

**Weighting:** 40-30-20-10 Hybrid Stratification
- 40% from Q4 (highest TADP - most complex)
- 30% from Q3
- 20% from Q2
- 10% from Q1 (zero TADP - negative examples)

---

**Author:** Luqman  
**Project:** AI6129 Pathogen Tracking System  
**Date:** January 2025

In [1]:
import re
import json
import random
import math
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from google.colab import drive

In [2]:
# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted


In [3]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS FOR YOUR ENVIRONMENT
# =============================================================================

# Directory paths
GROUND_TRUTH_FOLDER = "/content/drive/MyDrive/AI6129/ground_truth"
OUTPUT_FOLDER = "/content/drive/MyDrive/AI6129/assay/validation_splits"

# Assay fields to count for TADP calculation
ASSAY_FIELDS = [
    "serotype",
    "mlst",
    "spi",
    "amr",
    "plasmid",
    "snp",
    "virulence_genes"
]

# Total expected samples
TOTAL_SAMPLES = 227

# Holdout ratio (held out for final evaluation after GEPA optimisation)
HOLDOUT_RATIO = 0.20  # 20% = 45 samples

# Nested training split ratios (of working set after holdout)
# These create nested subsets: 10% ⊂ 20% ⊂ 30%
NESTED_TRAINING_RATIOS = {
    "split_10": 0.10,  # ~18 samples
    "split_20": 0.20,  # ~36 samples
    "split_30": 0.30,  # ~55 samples
}

# Hybrid stratification weights (favour complex examples)
# Q4 = highest TADP (most complex), Q1 = lowest/zero TADP (simplest/negative)
QUARTILE_WEIGHTS = {
    "Q4": 0.40,  # 40% from highest TADP quartile
    "Q3": 0.30,  # 30% from medium-high TADP
    "Q2": 0.20,  # 20% from medium-low TADP
    "Q1": 0.10,  # 10% from lowest/zero TADP (negative examples)
}

# Random seed for reproducibility
RANDOM_SEED = 42

## TADP Calculation Functions

These functions calculate the Total Assay Data Points (TADP) for each document.

In [4]:
def count_ast_entries(ast_data: List[dict]) -> int:
    """
    Count the number of antibiotic entries in AST data.

    Args:
        ast_data: List of AST dictionaries, each containing 'Antibiotics' list

    Returns:
        Total count of antibiotic entries
    """
    total = 0

    if not ast_data or not isinstance(ast_data, list):
        return 0

    for ast_entry in ast_data:
        if isinstance(ast_entry, dict):
            antibiotics = ast_entry.get("Antibiotics", [])
            if isinstance(antibiotics, list):
                total += len(antibiotics)

    return total


def count_assay_fields_in_isolate(isolate: dict) -> int:
    """
    Count assay data points in a single isolate entry.

    Args:
        isolate: Dictionary containing isolate data

    Returns:
        Count of assay data points
    """
    count = 0

    if not isinstance(isolate, dict):
        return 0

    # Count standard assay fields (list-based)
    for field in ASSAY_FIELDS:
        val = isolate.get(field, [])
        if isinstance(val, list):
            count += len(val)
        elif isinstance(val, str) and val and val.lower() != "null":
            count += 1

    # Count AST entries (nested structure)
    count += count_ast_entries(isolate.get("ast_data", []))

    return count


def calculate_tadp(ground_truth_data: dict) -> int:
    """
    Calculate Total Assay Data Points (TADP) from ground truth.

    Counts individual values from:
    - serotype, mlst, spi, amr, plasmid, snp, virulence_genes (list lengths)
    - ast_data (count of antibiotic entries)

    Sources:
    - isolates_with_linking (per-isolate assay data)
    - no_isolates_only_assayinformation (study-level assay data)

    Does NOT count isolate_without_linking (just IDs, no assay data).

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Total Assay Data Points count
    """
    total = 0

    # Count from isolates_with_linking
    isolates_with_linking = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(isolates_with_linking, list):
        for isolate in isolates_with_linking:
            total += count_assay_fields_in_isolate(isolate)

    # Count from no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and nioai:
        total += count_assay_fields_in_isolate(nioai)

    return total


def categorise_document_sections(ground_truth_data: dict) -> str:
    """
    Categorise document by which sections contain data.

    Categories:
    - IWL: Only isolates_with_linking has data
    - IWOL: Only isolate_without_linking has data
    - NIOAI: Only no_isolates_only_assayinformation has data
    - Combined categories for multiple sections (e.g., IWL+NIOAI)
    - EMPTY: No sections have data

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Category string
    """
    has_iwl = False
    has_iwol = False
    has_nioai = False

    # Check isolates_with_linking
    iwl = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(iwl, list) and len(iwl) > 0:
        has_iwl = True

    # Check isolate_without_linking
    iwol = ground_truth_data.get("isolate_without_linking", [])
    if isinstance(iwol, list) and len(iwol) > 0:
        has_iwol = True

    # Check no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and bool(nioai):
        has_nioai = True

    # Determine category
    categories = []
    if has_iwl:
        categories.append("IWL")
    if has_iwol:
        categories.append("IWOL")
    if has_nioai:
        categories.append("NIOAI")

    if not categories:
        return "EMPTY"
    elif len(categories) == 1:
        return categories[0]
    else:
        return "+".join(categories)

## Data Loading Functions

In [5]:
def extract_pmcid_from_filename(filename: str) -> Optional[str]:
    """
    Extract PMC ID from JSON filename.

    Expected patterns:
    - PMC1234567.json
    - PMC1234567_metadata.json
    - pmc1234567.json (case-insensitive)

    Args:
        filename: The JSON filename

    Returns:
        PMC ID string (e.g., "PMC1234567") or None if not found
    """
    pattern = r'(PMC\d+)'
    match = re.search(pattern, filename, re.IGNORECASE)

    if match:
        return match.group(1).upper()

    return None


def load_all_ground_truth(folder_path: str) -> Dict[str, dict]:
    """
    Load all JSON ground truth files and calculate TADP for each.

    Args:
        folder_path: Path to the ground truth folder

    Returns:
        Dictionary mapping PMCID to document data with TADP
    """
    ground_truth_dict = {}
    errors = []

    json_files = list(Path(folder_path).glob("*.json"))
    print(f"Found {len(json_files)} JSON files in folder")

    for gt_file in json_files:
        try:
            with open(gt_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Extract PMCID
            pmcid = data.get("pmcid") or extract_pmcid_from_filename(gt_file.name)

            if not pmcid:
                errors.append(f"Could not extract PMCID from: {gt_file.name}")
                continue

            # Calculate TADP
            tadp = calculate_tadp(data)
            category = categorise_document_sections(data)

            # Count isolates and other metrics
            iwl_count = len(data.get("isolates_with_linking", []))
            iwol_count = len(data.get("isolate_without_linking", []))
            clean_text_length = len(data.get("clean_text", ""))

            ground_truth_dict[pmcid] = {
                "pmcid": pmcid,
                "filename": gt_file.name,
                "filepath": str(gt_file),
                "tadp": tadp,
                "category": category,
                "isolates_with_linking_count": iwl_count,
                "isolate_without_linking_count": iwol_count,
                "clean_text_length": clean_text_length,
                "raw_data": data
            }

        except json.JSONDecodeError as e:
            errors.append(f"JSON parse error in {gt_file.name}: {e}")
        except Exception as e:
            errors.append(f"Error loading {gt_file.name}: {e}")

    if errors:
        print(f"\nWarning: {len(errors)} files had errors:")
        for err in errors[:10]:
            print(f"  - {err}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more")

    return ground_truth_dict

## TADP Distribution Analysis

In [6]:
def analyse_tadp_distribution(ground_truth_dict: Dict[str, dict]) -> dict:
    """
    Analyse the distribution of TADP across all documents.

    Args:
        ground_truth_dict: Dictionary from load_all_ground_truth()

    Returns:
        Statistics dictionary with distribution data
    """
    statistics = {
        "total_documents": len(ground_truth_dict),
        "tadp_values": [],
        "category_counts": Counter(),
        "zero_tadp_count": 0,
        "max_tadp": 0,
        "min_tadp": float('inf'),
        "total_tadp": 0,
        "mean_tadp": 0,
    }

    for pmcid, doc_data in ground_truth_dict.items():
        tadp = doc_data["tadp"]
        category = doc_data["category"]

        statistics["tadp_values"].append(tadp)
        statistics["category_counts"][category] += 1
        statistics["total_tadp"] += tadp

        if tadp == 0:
            statistics["zero_tadp_count"] += 1

        if tadp > statistics["max_tadp"]:
            statistics["max_tadp"] = tadp

        if tadp < statistics["min_tadp"]:
            statistics["min_tadp"] = tadp

    # Calculate mean
    if statistics["total_documents"] > 0:
        statistics["mean_tadp"] = (
            statistics["total_tadp"] / statistics["total_documents"]
        )

    # Calculate frequency distribution
    statistics["tadp_frequency"] = dict(Counter(statistics["tadp_values"]))

    return statistics


def print_tadp_distribution(statistics: dict) -> None:
    """
    Print TADP distribution summary.
    """
    print("\n" + "=" * 70)
    print("TADP DISTRIBUTION ANALYSIS")
    print("=" * 70)

    print(f"\nTotal Documents: {statistics['total_documents']}")
    print(f"Total Assay Data Points: {statistics['total_tadp']}")
    print(f"Mean TADP per Document: {statistics['mean_tadp']:.2f}")
    print(f"Min TADP: {statistics['min_tadp']}")
    print(f"Max TADP: {statistics['max_tadp']}")
    print(f"Documents with Zero TADP: {statistics['zero_tadp_count']}")

    print("\n" + "-" * 70)
    print("DOCUMENT CATEGORY DISTRIBUTION")
    print("-" * 70)

    total = statistics["total_documents"]
    for category, count in sorted(statistics["category_counts"].items(),
                                   key=lambda x: -x[1]):
        pct = (count / total * 100) if total > 0 else 0
        print(f"  {category:<20} {count:>5} ({pct:>5.1f}%)")

    print("\n" + "-" * 70)
    print("TADP FREQUENCY (first 20 values)")
    print("-" * 70)

    for tadp_val, freq in sorted(statistics["tadp_frequency"].items())[:20]:
        pct = (freq / total * 100) if total > 0 else 0
        print(f"  TADP={tadp_val:<5} {freq:>5} documents ({pct:>5.1f}%)")

## Quartile Assignment and Stratified Sampling

In [7]:
def assign_tadp_quartiles(documents: List[dict]) -> List[dict]:
    """
    Assign TADP quartile labels to documents.

    Sorts documents by TADP and assigns:
    - Q1: Lowest 25% (simplest, often TADP=0)
    - Q2: 25-50%
    - Q3: 50-75%
    - Q4: Highest 25% (most complex)

    Args:
        documents: List of document dictionaries with 'tadp' key

    Returns:
        Same list with 'quartile' key added to each document
    """
    # Sort by TADP (ascending)
    sorted_docs = sorted(documents, key=lambda x: x["tadp"])

    n = len(sorted_docs)
    quartile_size = n // 4
    remainder = n % 4

    # Calculate quartile boundaries
    # Distribute remainder to later quartiles (Q3, Q4 get extra if needed)
    boundaries = [0]
    current = 0
    for i in range(4):
        size = quartile_size + (1 if i >= (4 - remainder) else 0)
        current += size
        boundaries.append(current)

    # Assign quartiles
    for i, doc in enumerate(sorted_docs):
        if i < boundaries[1]:
            doc["quartile"] = "Q1"
        elif i < boundaries[2]:
            doc["quartile"] = "Q2"
        elif i < boundaries[3]:
            doc["quartile"] = "Q3"
        else:
            doc["quartile"] = "Q4"

    return sorted_docs


def print_quartile_summary(documents: List[dict]) -> None:
    """
    Print summary of quartile assignments.
    """
    print("\n" + "=" * 70)
    print("TADP QUARTILE SUMMARY")
    print("=" * 70)

    quartile_stats = {"Q1": [], "Q2": [], "Q3": [], "Q4": []}

    for doc in documents:
        quartile_stats[doc["quartile"]].append(doc["tadp"])

    print(f"\n{'Quartile':<10} {'Count':<8} {'Min TADP':<10} {'Max TADP':<10} {'Mean TADP':<12} {'Weight'}")
    print("-" * 70)

    for q in ["Q1", "Q2", "Q3", "Q4"]:
        values = quartile_stats[q]
        if values:
            min_v = min(values)
            max_v = max(values)
            mean_v = sum(values) / len(values)
            weight = QUARTILE_WEIGHTS[q]
            print(f"{q:<10} {len(values):<8} {min_v:<10} {max_v:<10} {mean_v:<12.1f} {weight:.0%}")

    print("\nNote: Q4 = most complex (highest TADP), Q1 = simplest (lowest/zero TADP)")

## Nested Training Split Creation

In [8]:
def create_stratified_holdout(
    documents: List[dict],
    holdout_ratio: float,
    random_seed: int
) -> Tuple[List[dict], List[dict]]:
    """
    Create stratified holdout set (20% for final evaluation).

    Stratifies by quartile to ensure balanced representation.

    Args:
        documents: List of documents with 'quartile' assigned
        holdout_ratio: Fraction to hold out (e.g., 0.20)
        random_seed: For reproducibility

    Returns:
        Tuple of (holdout_set, working_set)
    """
    random.seed(random_seed)

    # Group by quartile
    quartile_groups = {"Q1": [], "Q2": [], "Q3": [], "Q4": []}
    for doc in documents:
        quartile_groups[doc["quartile"]].append(doc)

    holdout_set = []
    working_set = []

    # Sample proportionally from each quartile
    for quartile, docs in quartile_groups.items():
        random.shuffle(docs)
        n_holdout = max(1, round(len(docs) * holdout_ratio))

        holdout_set.extend(docs[:n_holdout])
        working_set.extend(docs[n_holdout:])

    print(f"\nHoldout set: {len(holdout_set)} documents (stratified by quartile)")
    print(f"Working set: {len(working_set)} documents (available for GEPA training/validation)")

    return holdout_set, working_set


def create_nested_training_splits(
    working_set: List[dict],
    max_training_ratio: float,
    quartile_weights: Dict[str, float],
    random_seed: int
) -> Tuple[List[dict], Dict[str, List[str]]]:
    """
    Create nested training splits using hybrid stratification (40-30-20-10).

    The largest split (30%) is created first using weighted sampling,
    then smaller splits (20%, 10%) are nested subsets.

    Args:
        working_set: Documents available for training/validation
        max_training_ratio: Ratio for largest split (0.30 for 30%)
        quartile_weights: Dict mapping quartile to sampling weight
        random_seed: For reproducibility

    Returns:
        Tuple of (ordered_training_pool, split_indices)
        - ordered_training_pool: List of documents in nested order
        - split_indices: Dict mapping split name to list of PMCIDs
    """
    random.seed(random_seed)

    # Reassign quartiles to working set (since holdout removed some docs)
    working_set = assign_tadp_quartiles(working_set)

    # Group by quartile
    quartile_groups = {"Q1": [], "Q2": [], "Q3": [], "Q4": []}
    for doc in working_set:
        quartile_groups[doc["quartile"]].append(doc)

    # Shuffle within each quartile
    for q in quartile_groups:
        random.shuffle(quartile_groups[q])

    # Calculate total training pool size (30% of working set)
    total_training_size = round(len(working_set) * max_training_ratio)

    print(f"\nCreating training pool: {total_training_size} samples from {len(working_set)} working set")
    print(f"Using 40-30-20-10 hybrid stratification by TADP quartile")

    # Calculate samples per quartile using weights
    samples_per_quartile = {}
    allocated = 0

    for q in ["Q4", "Q3", "Q2", "Q1"]:  # Process in weight order
        weight = quartile_weights[q]
        target = round(total_training_size * weight)
        available = len(quartile_groups[q])
        actual = min(target, available)
        samples_per_quartile[q] = actual
        allocated += actual

    # Adjust if we didn't reach target (take from Q4 first, then Q3, etc.)
    shortfall = total_training_size - allocated
    if shortfall > 0:
        for q in ["Q4", "Q3", "Q2", "Q1"]:
            available_extra = len(quartile_groups[q]) - samples_per_quartile[q]
            take = min(shortfall, available_extra)
            samples_per_quartile[q] += take
            shortfall -= take
            if shortfall <= 0:
                break

    # Build ordered training pool (Q4 first, then Q3, Q2, Q1)
    # This ensures nested splits contain proportionally more complex examples
    ordered_training_pool = []

    for q in ["Q4", "Q3", "Q2", "Q1"]:
        n = samples_per_quartile[q]
        selected = quartile_groups[q][:n]
        ordered_training_pool.extend(selected)
        print(f"  {q}: {n} samples (target: {round(total_training_size * quartile_weights[q])}, available: {len(quartile_groups[q])})")

    print(f"  Total training pool: {len(ordered_training_pool)} samples")

    # Define nested split boundaries
    n_10 = round(len(working_set) * 0.10)  # 10% of working set
    n_20 = round(len(working_set) * 0.20)  # 20% of working set
    n_30 = len(ordered_training_pool)      # 30% of working set (full pool)

    # Create nested splits
    split_indices = {
        "split_10": [doc["pmcid"] for doc in ordered_training_pool[:n_10]],
        "split_20": [doc["pmcid"] for doc in ordered_training_pool[:n_20]],
        "split_30": [doc["pmcid"] for doc in ordered_training_pool[:n_30]],
    }

    print(f"\nNested split sizes:")
    print(f"  10% split: {len(split_indices['split_10'])} samples")
    print(f"  20% split: {len(split_indices['split_20'])} samples")
    print(f"  30% split: {len(split_indices['split_30'])} samples")
    print(f"\nVerification: 10% ⊂ 20% ⊂ 30%: {set(split_indices['split_10']).issubset(set(split_indices['split_20'])) and set(split_indices['split_20']).issubset(set(split_indices['split_30']))}")

    return ordered_training_pool, split_indices

## Split Summary and Validation

In [9]:
def print_split_summary(
    holdout_set: List[dict],
    working_set: List[dict],
    split_indices: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict]
) -> None:
    """
    Print comprehensive summary of all splits.
    """
    print("\n" + "=" * 70)
    print("COMPLETE SPLIT SUMMARY")
    print("=" * 70)

    # Holdout summary
    print(f"\nHOLDOUT SET: {len(holdout_set)} documents")
    print("  (Reserved for final evaluation after GEPA optimisation)")
    holdout_tadps = [d["tadp"] for d in holdout_set]
    print(f"  TADP range: {min(holdout_tadps)} - {max(holdout_tadps)}")
    print(f"  Mean TADP: {sum(holdout_tadps)/len(holdout_tadps):.1f}")
    quartile_dist = Counter(d["quartile"] for d in holdout_set)
    print(f"  Quartile distribution: {dict(sorted(quartile_dist.items()))}")

    # Working set summary
    print(f"\nWORKING SET: {len(working_set)} documents")
    print("  (Available for GEPA training and validation)")

    # Nested training splits
    print("\n" + "-" * 70)
    print("NESTED TRAINING CONFIGURATIONS")
    print("-" * 70)

    for split_name in ["split_10", "split_20", "split_30"]:
        pmcids = split_indices[split_name]
        n_train = len(pmcids)
        n_val = len(working_set) - n_train

        # Calculate TADP stats for this split
        tadps = [ground_truth_dict[p]["tadp"] for p in pmcids]

        pct = split_name.replace("split_", "")
        print(f"\n{pct}% Configuration:")
        print(f"  Training:   {n_train:>4} samples")
        print(f"  Validation: {n_val:>4} samples")
        print(f"  Training TADP range: {min(tadps)} - {max(tadps)}")
        print(f"  Training mean TADP: {sum(tadps)/len(tadps):.1f}")

        # Quartile distribution in training set
        q_dist = Counter(ground_truth_dict[p].get("quartile", "?") for p in pmcids)
        # Note: quartile may not be set in ground_truth_dict, need to recalculate
        print(f"  Training quartile dist: Q4={q_dist.get('Q4', 0)}, Q3={q_dist.get('Q3', 0)}, Q2={q_dist.get('Q2', 0)}, Q1={q_dist.get('Q1', 0)}")

## Output Functions

In [10]:
def save_split_assignments(
    holdout_set: List[dict],
    working_set: List[dict],
    split_indices: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict],
    output_folder: str,
    filename: str = "assay_tadp_nested_splits.json"
) -> str:
    """
    Save split assignments to JSON file for reproducibility.

    Args:
        holdout_set: List of holdout documents
        working_set: List of working set documents
        split_indices: Dict mapping split name to list of PMCIDs
        ground_truth_dict: Full ground truth dictionary
        output_folder: Path to output folder
        filename: Output filename

    Returns:
        Path to saved file
    """
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename

    # Build comprehensive output
    output_data = {
        "metadata": {
            "created_date": datetime.now().isoformat(),
            "random_seed": RANDOM_SEED,
            "total_samples": len(ground_truth_dict),
            "holdout_ratio": HOLDOUT_RATIO,
            "quartile_weights": QUARTILE_WEIGHTS,
            "stratification_metric": "TADP (Total Assay Data Points)",
            "split_structure": "Nested (10% ⊂ 20% ⊂ 30%)",
            "assay_fields_counted": ASSAY_FIELDS
        },
        "holdout": {
            "pmcids": [d["pmcid"] for d in holdout_set],
            "count": len(holdout_set),
            "purpose": "Final evaluation after GEPA optimisation"
        },
        "working_set": {
            "pmcids": [d["pmcid"] for d in working_set],
            "count": len(working_set),
            "purpose": "Available for GEPA training and validation"
        },
        "nested_training_splits": {
            "split_10": {
                "pmcids": split_indices["split_10"],
                "count": len(split_indices["split_10"]),
                "validation_count": len(working_set) - len(split_indices["split_10"]),
                "description": "10% training split (minimum viable)"
            },
            "split_20": {
                "pmcids": split_indices["split_20"],
                "count": len(split_indices["split_20"]),
                "validation_count": len(working_set) - len(split_indices["split_20"]),
                "description": "20% training split (conservative)"
            },
            "split_30": {
                "pmcids": split_indices["split_30"],
                "count": len(split_indices["split_30"]),
                "validation_count": len(working_set) - len(split_indices["split_30"]),
                "description": "30% training split (generous)"
            }
        },
        "document_details": {}
    }

    # Add document details
    for pmcid, doc_data in ground_truth_dict.items():
        # Determine assignment
        if pmcid in [d["pmcid"] for d in holdout_set]:
            assignment = "holdout"
        elif pmcid in split_indices["split_10"]:
            assignment = "training_10_20_30"
        elif pmcid in split_indices["split_20"]:
            assignment = "training_20_30"
        elif pmcid in split_indices["split_30"]:
            assignment = "training_30"
        else:
            assignment = "validation_only"

        output_data["document_details"][pmcid] = {
            "tadp": doc_data["tadp"],
            "category": doc_data["category"],
            "assignment": assignment,
            "isolates_with_linking_count": doc_data["isolates_with_linking_count"],
            "isolate_without_linking_count": doc_data["isolate_without_linking_count"],
            "clean_text_length": doc_data["clean_text_length"]
        }

    # Save
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)

    print(f"\nSplit assignments saved to: {output_path}")
    return str(output_path)


def export_to_csv(
    holdout_set: List[dict],
    working_set: List[dict],
    split_indices: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict],
    output_folder: str,
    filename: str = "assay_tadp_nested_splits.csv"
) -> str:
    """
    Export split assignments to CSV for easy viewing.
    """
    import csv

    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename

    # Build assignment lookup
    holdout_pmcids = set(d["pmcid"] for d in holdout_set)

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        writer.writerow([
            "PMCID",
            "TADP",
            "Category",
            "Assignment",
            "In_10pct",
            "In_20pct",
            "In_30pct",
            "IWL_Count",
            "IWOL_Count"
        ])

        for pmcid in sorted(ground_truth_dict.keys()):
            doc = ground_truth_dict[pmcid]

            if pmcid in holdout_pmcids:
                assignment = "holdout"
            elif pmcid in split_indices["split_30"]:
                assignment = "training"
            else:
                assignment = "validation"

            in_10 = "Y" if pmcid in split_indices["split_10"] else "N"
            in_20 = "Y" if pmcid in split_indices["split_20"] else "N"
            in_30 = "Y" if pmcid in split_indices["split_30"] else "N"

            writer.writerow([
                pmcid,
                doc["tadp"],
                doc["category"],
                assignment,
                in_10,
                in_20,
                in_30,
                doc["isolates_with_linking_count"],
                doc["isolate_without_linking_count"]
            ])

    print(f"CSV export saved to: {output_path}")
    return str(output_path)

## Main Execution

In [11]:
def main():
    """
    Main execution function.

    Creates TADP-stratified nested training splits for GEPA experiments.
    """
    print("=" * 70)
    print("ASSAY GROUND TRUTH: TADP STRATIFICATION WITH NESTED TRAINING SPLITS")
    print("=" * 70)
    print(f"\nDesign: 20% holdout + nested 10%⊂20%⊂30% training splits")
    print(f"Stratification: TADP with 40-30-20-10 quartile weighting")

    # Step 1: Load ground truth
    print("\n" + "-" * 70)
    print("STEP 1: Loading Ground Truth")
    print("-" * 70)
    print(f"Loading from: {GROUND_TRUTH_FOLDER}")

    ground_truth_dict = load_all_ground_truth(GROUND_TRUTH_FOLDER)

    if not ground_truth_dict:
        print("ERROR: No ground truth files loaded.")
        return None

    print(f"Loaded {len(ground_truth_dict)} documents")

    # Step 2: Analyse TADP distribution
    print("\n" + "-" * 70)
    print("STEP 2: Analysing TADP Distribution")
    print("-" * 70)

    statistics = analyse_tadp_distribution(ground_truth_dict)
    print_tadp_distribution(statistics)

    # Step 3: Assign quartiles
    print("\n" + "-" * 70)
    print("STEP 3: Assigning TADP Quartiles")
    print("-" * 70)

    documents = list(ground_truth_dict.values())
    documents = assign_tadp_quartiles(documents)

    # Update ground_truth_dict with quartile assignments
    for doc in documents:
        ground_truth_dict[doc["pmcid"]]["quartile"] = doc["quartile"]

    print_quartile_summary(documents)

    # Step 4: Create stratified holdout
    print("\n" + "-" * 70)
    print("STEP 4: Creating Stratified Holdout (20%)")
    print("-" * 70)

    holdout_set, working_set = create_stratified_holdout(
        documents,
        HOLDOUT_RATIO,
        RANDOM_SEED
    )

    # Step 5: Create nested training splits
    print("\n" + "-" * 70)
    print("STEP 5: Creating Nested Training Splits (10% ⊂ 20% ⊂ 30%)")
    print("-" * 70)

    training_pool, split_indices = create_nested_training_splits(
        working_set,
        max_training_ratio=0.30,
        quartile_weights=QUARTILE_WEIGHTS,
        random_seed=RANDOM_SEED
    )

    # Step 6: Print comprehensive summary
    print_split_summary(holdout_set, working_set, split_indices, ground_truth_dict)

    # Step 7: Save outputs
    print("\n" + "-" * 70)
    print("STEP 7: Saving Split Assignments")
    print("-" * 70)

    save_split_assignments(
        holdout_set, working_set, split_indices,
        ground_truth_dict, OUTPUT_FOLDER
    )

    export_to_csv(
        holdout_set, working_set, split_indices,
        ground_truth_dict, OUTPUT_FOLDER
    )

    print("\n" + "=" * 70)
    print("SPLIT CREATION COMPLETE")
    print("=" * 70)

    return {
        "ground_truth_dict": ground_truth_dict,
        "statistics": statistics,
        "holdout_set": holdout_set,
        "working_set": working_set,
        "split_indices": split_indices
    }

In [12]:
if __name__ == "__main__":
    results = main()

ASSAY GROUND TRUTH: TADP STRATIFICATION WITH NESTED TRAINING SPLITS

Design: 20% holdout + nested 10%⊂20%⊂30% training splits
Stratification: TADP with 40-30-20-10 quartile weighting

----------------------------------------------------------------------
STEP 1: Loading Ground Truth
----------------------------------------------------------------------
Loading from: /content/drive/MyDrive/AI6129/ground_truth
Found 227 JSON files in folder
Loaded 227 documents

----------------------------------------------------------------------
STEP 2: Analysing TADP Distribution
----------------------------------------------------------------------

TADP DISTRIBUTION ANALYSIS

Total Documents: 227
Total Assay Data Points: 4737
Mean TADP per Document: 20.87
Min TADP: 0
Max TADP: 292
Documents with Zero TADP: 92

----------------------------------------------------------------------
DOCUMENT CATEGORY DISTRIBUTION
----------------------------------------------------------------------
  IWL             

## Verification: Check Nested Structure

In [13]:
# Verify nested structure
if results:
    split_10 = set(results["split_indices"]["split_10"])
    split_20 = set(results["split_indices"]["split_20"])
    split_30 = set(results["split_indices"]["split_30"])

    print("Nested Structure Verification:")
    print(f"  10% ⊂ 20%: {split_10.issubset(split_20)}")
    print(f"  20% ⊂ 30%: {split_20.issubset(split_30)}")
    print(f"  10% ⊂ 30%: {split_10.issubset(split_30)}")

    # Show what's added at each level
    print(f"\nSamples unique to each level:")
    print(f"  10% split: {len(split_10)} samples")
    print(f"  20% split adds: {len(split_20 - split_10)} samples")
    print(f"  30% split adds: {len(split_30 - split_20)} samples")

Nested Structure Verification:
  10% ⊂ 20%: True
  20% ⊂ 30%: True
  10% ⊂ 30%: True

Samples unique to each level:
  10% split: 18 samples
  20% split adds: 19 samples
  30% split adds: 18 samples


## Optional: View Sample Documents by Split

In [14]:
# Uncomment to view sample documents
if results:
    print("\nSample documents in 10% training split:")
    for pmcid in results["split_indices"]["split_10"][:5]:
        doc = results["ground_truth_dict"][pmcid]
        print(f"  {pmcid}: TADP={doc['tadp']}, Category={doc['category']}")


Sample documents in 10% training split:
  PMC7022574: TADP=25, Category=IWL+IWOL
  PMC6412060: TADP=30, Category=IWL
  PMC6430326: TADP=29, Category=IWL
  PMC4881965: TADP=107, Category=IWL
  PMC9670943: TADP=28, Category=IWL
